In [1]:
import os
from dotenv import load_dotenv
import wrds
import pandas as pd
import yfinance as yf
from collections import Counter

# comp.idxcst_his = historical index constituents (gvkeyx '000003' = S&P 500)
# comp.secd = security daily (price, adjustment factors)
db = wrds.Connection(wrds_username=os.getenv("WRDS_USERNAME"))
output_dir = "data_raw"
os.makedirs(output_dir, exist_ok=True)


start_date = "2000-01-01"
end_date = "2026-04-01"


# Step 1: Parse CSV to get all unique constituent tickers since 2000
print("Parsing historical constituent CSV...")
df_hist = pd.read_csv("S&P 500 Historical Components & Changes(01-17-2026).csv")
df_hist["date"] = pd.to_datetime(df_hist["date"])
df_hist = df_hist[df_hist["date"] >= start_date].copy()

all_tickers = set()
for row in df_hist["tickers"]:
    all_tickers.update(row.split(","))
all_tickers = sorted(all_tickers)
print(f"  Unique tickers ever in S&P 500 since {start_date}: {len(all_tickers)}")

tickers_sql = "', '".join(all_tickers)

# Step 2: Map tickers → PERMNOs via crsp.msenames
# namedt/nameendt define when this ticker was valid for this PERMNO
# (handles ticker reuse across different companies over time)
print("Mapping tickers to PERMNOs via crsp.msenames...")
sql_names = f"""
SELECT
    permno,
    ticker,
    comnam,
    namedt,
    nameendt
FROM crsp.msenames
WHERE ticker IN ('{tickers_sql}')
  AND nameendt >= '{start_date}'
ORDER BY permno, namedt
"""
df_names = db.raw_sql(sql_names, date_cols=["namedt", "nameendt"])
print(f"  Ticker-PERMNO mappings: {len(df_names)}")
print(f"  Unique PERMNOs:         {df_names['permno'].nunique()}")

# Most recent ticker per PERMNO — used to label columns in the returns matrix
df_names_latest = df_names.sort_values("namedt").groupby("permno").last().reset_index()[["permno", "ticker", "comnam"]]
permnos = tuple(int(p) for p in df_names["permno"].unique())

# Step 3: Pull daily returns from crsp.dsf
# ret   = total holding period return (price change + dividends), already adjusted
# prc   = closing price; CRSP uses negative values for bid-ask midpoints → take ABS
# shrout = shares outstanding in thousands
print(f"Fetching daily returns from crsp.dsf for {len(permnos):,} PERMNOs...")
sql_dsf = f"""
SELECT
    permno,
    date,
    ret,
    ABS(prc)   AS prc,
    shrout,
    vol
FROM crsp.dsf
WHERE permno IN {permnos}
  AND date >= '{start_date}'
  AND date <= '{end_date}'
ORDER BY date, permno
"""
df_dsf = db.raw_sql(sql_dsf, date_cols=["date"])
print(f"  Raw rows fetched: {len(df_dsf):,}")

# Step 4: Pivot to wide returns matrix (trading days × stocks)
df_dsf["date"] = pd.to_datetime(df_dsf["date"]).dt.normalize()

# Pivot using permno (always unique) — avoids duplicate ticker collisions
df_returns_matrix = df_dsf.pivot_table(index="date", columns="permno", values="ret", aggfunc="first")
df_returns_matrix.columns.name = None
df_returns_matrix.columns = df_returns_matrix.columns.astype(int)

# Build a deduplicated ticker label: suffix with permno when a ticker is reused
permno_to_ticker = dict(zip(df_names_latest["permno"], df_names_latest["ticker"]))

ticker_freq = Counter(permno_to_ticker.values())
col_labels = {permno: (f"{ticker}_{permno}" if ticker_freq[ticker] > 1 else ticker) for permno, ticker in permno_to_ticker.items()}
df_returns_matrix = df_returns_matrix.rename(columns=col_labels)

print(f"\nReturns matrix shape:          {df_returns_matrix.shape}")
print(f"  Date range:                  {df_returns_matrix.index.min().date()} to {df_returns_matrix.index.max().date()}")
print(f"  Unique stocks (ever-constituent):  {df_returns_matrix.shape[1]}")
print(f"  Avg active stocks per day:   {df_returns_matrix.notna().sum(axis=1).mean():.0f}")
print(f"  Overall NaN density:         {df_returns_matrix.isna().mean().mean():.1%}")

# Step 5: Save outputs
df_returns_matrix.to_parquet(f"{output_dir}/sp500_returns_matrix.parquet", engine="pyarrow", compression="snappy")
# Constituent history — use this to mask returns to actual membership windows
# during rolling RMT analysis: intersect column tickers with CSV for each window
df_hist.to_parquet(f"{output_dir}/sp500_constituent_history.parquet", engine="pyarrow", compression="snappy", index=False)
df_names.to_parquet(f"{output_dir}/permno_ticker_map.parquet", engine="pyarrow", compression="snappy", index=False)
print("\nSaved: sp500_returns_matrix.parquet")
print("Saved: sp500_constituent_history.parquet")
print("Saved: permno_ticker_map.parquet")

Loading library list...
Done
Parsing historical constituent CSV...
  Unique tickers ever in S&P 500 since 2000-04-01: 1074
Mapping tickers to PERMNOs via crsp.msenames...
  Ticker-PERMNO mappings: 5420
  Unique PERMNOs:         1305
Fetching daily returns from crsp.dsf for 1,305 PERMNOs...
  Raw rows fetched: 5,061,795

Returns matrix shape:          (6226, 1305)
  Date range:                  2000-04-03 to 2024-12-31
  Unique stocks (ever-constituent):  1305
  Avg active stocks per day:   809
  Overall NaN density:         38.0%

Saved: sp500_returns_matrix.parquet
Saved: sp500_constituent_history.parquet
Saved: permno_ticker_map.parquet
